# 后效大盘分布分析 —— 内循环后效

**订单数据来源**：`ks_ad_antispam.ad_merchant_order_wide_feature_base_di`  
**消耗数据来源**：`ad_rc_data.ad_kuaishou_account_visitor_stat_di`  
**ocpc_action_type**：`AD_MERCHANT_ROAS / EVENT_ORDER_PAIED / AD_STOREWIDE_ROAS / AD_MERCHANT_T7_ROI / AD_FANS_TOP_ROI`  
**分析维度**：`visitor_id`  
**分箱方式**：首尾各1个尾部开区间（p5以下 / p95以上），中间8个等距分箱，共10箱  
**指标**：gmv / refund_gmv / order_cnt / refund_cnt / avg_price / avg_net_price / refund_price / cost_yuan / spam_cost_yuan / total_cost_yuan / roi / roi_total

---

## Step 1：配置（修改 P_DATE 后运行）

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

P_DATE = '20250101'

INCYCLE_OCPC_TYPES = "'EVENT_ORDER_PAIED','AD_MERCHANT_T7_ROI','AD_STOREWIDE_ROAS','AD_MERCHANT_ROAS','AD_FANS_TOP_ROI'"

## Step 2：确认 Spark（KML 平台已预注入 spark，无需手动创建）

In [ ]:
print(spark)
print('Spark version:', spark.version)

## Step 3：取订单数据（order_info）

In [ ]:
sql_order = f"""
SELECT
    visitor_id,
    SUM(order_product_payment_amt) / 100000.0                                                     AS gmv,
    SUM(IF(is_refund > 0 AND refund_type <> 2, order_product_payment_amt, 0)) / 100000.0          AS refund_gmv,
    CAST(COUNT(*) AS DOUBLE)                                                                       AS order_cnt,
    CAST(SUM(IF(is_refund > 0 AND refund_type <> 2, 1, 0)) AS DOUBLE)                             AS refund_cnt
FROM ks_ad_antispam.ad_merchant_order_wide_feature_base_di
WHERE p_date = '{P_DATE}'
  AND attribution_type = 1
  AND resource_type <> 'UNION'
  AND ocpc_action_type IN ({INCYCLE_OCPC_TYPES})
GROUP BY visitor_id
"""

print('正在取订单数据...')
df_order = spark.sql(sql_order).toPandas()
print(f'订单数据 visitor_id 数: {len(df_order):,}')
df_order.head()

## Step 4：取消耗数据（cost_info）

In [ ]:
sql_cost = f"""
SELECT
    visitor_id,
    SUM(cost_yuan)                      AS cost_yuan,
    SUM(spam_cost_yuan)                 AS spam_cost_yuan,
    SUM(cost_yuan + spam_cost_yuan)     AS total_cost_yuan
FROM ad_rc_data.ad_kuaishou_account_visitor_stat_di
WHERE p_date = '{P_DATE}'
  AND ocpc_action_type IN ({INCYCLE_OCPC_TYPES})
GROUP BY visitor_id
"""

print('正在取消耗数据...')
df_cost = spark.sql(sql_cost).toPandas()
print(f'消耗数据 visitor_id 数: {len(df_cost):,}')
df_cost.head()

## Step 5：合并，计算合成指标

In [ ]:
df = pd.merge(df_order, df_cost, on='visitor_id', how='outer')

df['avg_price']     = df.apply(lambda r: r['gmv'] / r['order_cnt']                    if r['order_cnt'] > 0 else None, axis=1)
df['avg_net_price'] = df.apply(lambda r: (r['gmv'] - r['refund_gmv']) / r['order_cnt'] if r['order_cnt'] > 0 else None, axis=1)
df['refund_price']  = df.apply(lambda r: r['refund_gmv'] / r['order_cnt']              if r['order_cnt'] > 0 else None, axis=1)
df['roi']           = df.apply(lambda r: (r['gmv'] - r['refund_gmv']) / r['cost_yuan']       if (r['cost_yuan'] is not None and r['cost_yuan'] > 0) else None, axis=1)
df['roi_total']     = df.apply(lambda r: (r['gmv'] - r['refund_gmv']) / r['total_cost_yuan'] if (r['total_cost_yuan'] is not None and r['total_cost_yuan'] > 0) else None, axis=1)

print(f'合并后总 visitor_id 数: {len(df):,}')
df.describe()

## Step 6：工具函数

In [ ]:
def tail_equal_width_distribution(df, value_col, n_mid=8):
    s = df[value_col].dropna()
    s = s[s > 0]
    if s.empty:
        print(f'  [WARN] {value_col}: 无有效数据')
        return pd.DataFrame()

    p5  = s.quantile(0.05)
    p95 = s.quantile(0.95)

    mid_edges = np.linspace(p5, p95, n_mid + 1)
    edges = np.unique(np.concatenate([[0], mid_edges, [np.inf]]))

    labels = []
    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i + 1]
        lo_str = '0' if lo == 0 else f'{lo:.4f}'
        hi_str = '+inf' if hi == np.inf else f'{hi:.4f}'
        labels.append(f'({lo_str}, {hi_str}]')

    bins = pd.cut(s, bins=edges, labels=labels, include_lowest=True, right=True)
    result = (
        s.groupby(bins, observed=True)
        .agg(cnt='count', min_val='min', max_val='max', avg_val='mean', total_val='sum')
        .reset_index()
    )
    result.columns = ['分箱区间', '用户数cnt', '最小值', '最大值', '均值', '总量']
    result['占比%'] = (result['用户数cnt'] / result['用户数cnt'].sum() * 100).round(2)
    result['p5边界']  = round(p5,  4)
    result['p95边界'] = round(p95, 4)
    return result


def show_distribution(result, metric_name, save_csv=True):
    print(f"\n{'='*70}")
    print(f'  {metric_name}  (visitor_id维度，尾部开区间+中间等距，共10箱)')
    print(f"{'='*70}")
    if result.empty:
        return
    print(result.to_string(index=False))
    if save_csv:
        fname = f'dist_incycle_{metric_name}.csv'
        result.to_csv(fname, index=False)
        print(f'  => 已保存: {fname}')

## Step 7：各指标分布分析

### 7.1 GMV（gmv）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'gmv'), 'gmv')

### 7.2 退款GMV（refund_gmv）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'refund_gmv'), 'refund_gmv')

### 7.3 订单数（order_cnt）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'order_cnt'), 'order_cnt')

### 7.4 退款单数（refund_cnt）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'refund_cnt'), 'refund_cnt')

### 7.5 客单价 avg_price（不扣退款）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'avg_price'), 'avg_price')

### 7.6 平均成单金额 avg_net_price（扣退款）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'avg_net_price'), 'avg_net_price')

### 7.7 退款均价（refund_price）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'refund_price'), 'refund_price')

### 7.8 正常流量消耗（cost_yuan）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'cost_yuan'), 'cost_yuan')

### 7.9 作弊流量消耗（spam_cost_yuan）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'spam_cost_yuan'), 'spam_cost_yuan')

### 7.10 总消耗（total_cost_yuan）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'total_cost_yuan'), 'total_cost_yuan')

### 7.11 净ROI（roi，不含作弊消耗）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'roi'), 'roi')

### 7.12 净ROI（roi_total，含作弊消耗）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'roi_total'), 'roi_total')

## Step 8：汇总总览（p50 / p90 / p99）

In [ ]:
metric_cols = [
    'gmv', 'refund_gmv', 'order_cnt', 'refund_cnt',
    'avg_price', 'avg_net_price', 'refund_price',
    'cost_yuan', 'spam_cost_yuan', 'total_cost_yuan',
    'roi', 'roi_total'
]

summary = []
for col in metric_cols:
    s = df[col].dropna()
    s = s[s > 0]
    if s.empty:
        summary.append({'口径': col, '有效UV数': 0})
        continue
    summary.append({
        '口径': col,
        '有效UV数': len(s),
        '均值': round(s.mean(), 4),
        'p50': round(s.quantile(0.5), 4),
        'p90': round(s.quantile(0.9), 4),
        'p99': round(s.quantile(0.99), 4),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv('incycle_summary.csv', index=False)
print('\n=> 已保存: incycle_summary.csv')